In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="NETSUITE",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="fact_financial_pnl",
    choices=["discount", "sales", "fact_financial_pnl", "fact_financial_pnl"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
%sql
select * from fq_dev_catalog.bronze.gl_report limit 1

In [0]:
%run ./src/foodquest_pnl

In [0]:
df = spark.read.option('multiline', False).format('json').load(f'{staging}/FoodQuest/Netsuite/Wastage/ALBAIK/2026/Feb/wastage*.json')
exploded_df = (
            df.select(
                explode('results').alias('result')
            ).select('result.*')
        )
# display(exploded_df)


df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master")
df_location_master = spark.read.table("fq_dev_catalog.silver.dim_location_master")

# Step 1: Join both master tables
df_all_masters = exploded_df.join(
    df_coa_master, 
    df_coa_master["account_number"].cast("string") == exploded_df["accountNumber"], 
    'inner'
).join(
    df_location_master,
    col("location") == df_location_master.netsuite_location_name,
    'left'
)

df_final_netsuite = final_df(df_all_masters, 'actual')
df_final_netsuite.display()

In [0]:
df = spark.read.option('multiline', False).format('json').load(f'{staging}/response*.json')
exploded_df = (
            df.select(
                explode('results').alias('result')
            ).select('result.*')
        )
# display(exploded_df)


df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master")
df_location_master = spark.read.table("fq_dev_catalog.silver.dim_location_master")

# Step 1: Join both master tables
df_all_masters = exploded_df.join(
    df_coa_master, 
    df_coa_master["account_number"].cast("string") == exploded_df["accountNo"], 
    'inner'
).join(
    df_location_master,
    col("location") == df_location_master.netsuite_location_name,
    'left'
)

df_final = final_df(df_all_masters, "actual")
df_final.display()

In [0]:

df = spark.read.table('fq_dev_catalog.bronze.pnl_actual_flat_data')

from pyspark.sql.functions import *
df_filtered = df.filter(((col("Store Name")=="Dubai Mall") & (col("File Name")=="February") & (col("Year")==2026)) 
                        # | ((col("Store Name")=="Dubai Mall") & (col("File Name")=="January") & (col("Year")==2026)) | ((col("Store Name")=="DEN DCC") & (col("File Name")=="January") & (col("Year")==2025)) | ((col("Store Name")=="DEN DCC") & (col("File Name")=="January") & (col("Year")==2026))
)


df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master")
df_location_master = spark.read.table("fq_dev_catalog.silver.dim_location_master")
df_coa_master_distinct = df_coa_master.dropDuplicates(["mapped_name"]) # Distinct new_grouping


# Step 1: Joined both master tables upfront
df_all_masters = df_filtered.join(
    df_coa_master_distinct, 
    df_coa_master_distinct["mapped_name"].cast("string") == df_filtered["Column 1"], 
    'inner'
).join(
    df_location_master,
    col("Store name") == df_location_master["excel_p&l_name"],
    'left'
)

df_all_masters = df_all_masters.withColumnsRenamed(
    {
        "File name": "month",
        "Year": "year",
        # "Column 1": "account_name",
        "Act": "amount",
        "Store name": "location"
    }
)

df_final = final_df(df_all_masters, "actual")
df_final.display()

In [0]:
df = spark.read.table('fq_dev_catalog.bronze.pnl_budget_flat_data')

from pyspark.sql.functions import *
df_filtered = df.filter(((col("Store Name")=="Dubai Mall") & (col("File Name")=="February") & (col("Year")==2026)) )
                        #  | ((col("Store Name")=="Dubai Mall") & (col("File Name")=="January") & (col("Year")==2026)) | ((col("Store Name")=="DEN DCC") & (col("File Name")=="January") & (col("Year")==2025)) | ((col("Store Name")=="DEN DCC") & (col("File Name")=="January") & (col("Year")==2026)))
# df_filtered.display()

df_coa_master = spark.read.table("fq_dev_catalog.silver.dim_coa_master")
df_location_master = spark.read.table("fq_dev_catalog.silver.dim_location_master")
df_coa_master_distinct = df_coa_master.dropDuplicates(["mapped_name"]) # Distinct new_grouping

# Step 1: Join both master tables upfront
df_all_masters = df_filtered.join(
    df_coa_master_distinct, 
    df_coa_master_distinct["mapped_name"].cast("string") == df_filtered["Column 1"], 
    'inner'
).join(
    df_location_master,
    col("Store name") == df_location_master["excel_p&l_name"],
    'left'
)

df_all_masters = df_all_masters.withColumnsRenamed(
{
    "File name": "month",
    "Year": "year",
    # "Column 1": "account_name",
    "Bud": "amount",
    "Store name": "location"
}
)


df_final_budget = final_df(df_all_masters, "budget")
df_final_budget.display()

In [0]:
df_coa_master.select('account_number','account_name', 'name','accoun_type','majour_group', 'group', 'sub_group', 'detail/total','calculation_type','sort_order').join(df_final_netsuite.filter(col('netsuite_location_name') == 'ALB-100-Dubai Mall'), "account_name", "left").join(df_final_budget.filter(col('netsuite_location_name') == 'ALB-100-Dubai Mall'), "account_name", "left").display(  )

In [0]:
df_final_with_forecast = df_final_snake.withColumn(
    "final_forecast_amount",
    when(col("actual_amount").isNotNull(), col("actual_amount"))
    # .when(col("forecaste").isNotNull(), col("forecaste"))
    .when(col("budget_amount").isNotNull(), col("budget_amount"))
    .otherwise(lit(0.0))  # or lit(None) if you prefer null
)

In [0]:
matching_columns = [col for col in df1.columns if col in df2.columns]

In [0]:
%sql
CREATE EXTERNAL TABLE IF NOT EXISTS fq_dev_catalog.silver.fact_financial_pnl (
  account_name STRING,
  account_number STRING,
  majour_group STRING,
  group_name STRING,
  sub_group STRING,
  alternate_group STRING,
  account_type STRING,
  location STRING,
  type STRING COMMENT 'HO/Store',
  location_code STRING,
  brand STRING,
  company STRING,
  cluster STRING,
  location_mode STRING COMMENT 'Mall/Drive thru/Stand alone',
  emirates STRING,
  detail_total_grand_total STRING,
  sum_order INT,
  actual_value DECIMAL(18,2),
  budget DECIMAL(18,2),
  previous_year_sales DECIMAL(18,2),
  year INT,
  month INT,
  forecast DECIMAL(18,2),
  calculation_type STRING,
  actual_net_sales DECIMAL(18,2),
  budget_net_sales DECIMAL(18,2),
  py_net_sales DECIMAL(18,2),
  brand_act_net_sales DECIMAL(18,2),
  brand_budget_net_sales DECIMAL(18,2),
  brand_py_net_sales DECIMAL(18,2),
  company_act_net_sales DECIMAL(18,2),
  company_budget_net_sales DECIMAL(18,2),
  company_py_net_sales DECIMAL(18,2)
)
USING DELTA
CLUSTER BY (year, month, company, brand)
LOCATION 'abfss://fq-dev-silver-container@fqadfstoragedev.dfs.core.windows.net/external/fact_financial_pnl' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
def merge_stream_fact_financial_pnl(df, i):
    try:
        exploded_df = (
            df.select(
                explode('results').alias('result')
            ).select('result.*')
        )
        
        fact_financial_pnl_upsert = enrich_json(exploded_df)
        fact_financial_pnl_upsert.createOrReplaceTempView("fact_financial_pnl_upsert_microbatch")
       
        df.sparkSession.sql("""
            MERGE INTO fq_dev_catalog.silver.fact_financial_pnl target
            USING (
                SELECT *
                FROM fact_financial_pnl_upsert_microbatch
            ) as source
            ON target.year = source.year
                AND target.month = source.month
                AND target.location = source.location
                AND target.account_name = source.account_name
                AND target.sum_order = source.sum_order
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        
        print(f"Successfully merged batch {i}")

    # df.sparkSession.sql("""
    #     MERGE INTO fq_dev_catalog.silver.fact_financial_pnl target
    #     USING (
    #         SELECT *
    #         FROM (
    #             SELECT *, 
    #                 ROW_NUMBER() OVER (
    #                     PARTITION BY year, month, location, account_name, sum_order
    #                     ORDER BY year DESC  -- or add a load_time column
    #                 ) as rank
    #             FROM fact_financial_pnl_upsert_microbatch
    #         )
    #         WHERE rank = 1
    #     ) as source
    #     ON target.year = source.year
    #         AND target.month = source.month
    #         AND target.location = source.location
    #         AND target.account_name = source.account_name
    #         AND target.sum_order = source.sum_order
    #     WHEN MATCHED THEN UPDATE SET *
    #     WHEN NOT MATCHED THEN INSERT *
    # """)
    except Exception as e:
        print(f"Error in merge_stream: {e}")
        raise e

(spark.readStream
    # .option("schemaTrackingLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpointing_fact_financial_pnl/schema_fact_financial_pnl')
    .table("fq_dev_catalog.bronze.gl_report")
    .writeStream
    .foreachBatch(merge_stream_fact_financial_pnl)
    .option("mergeSchema", "true")
    .option('skipChangeCommits', "true")
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_silver_fact_financial_pnl3')  
    .trigger(availableNow=True)
    .start()
).awaitTermination()

time.sleep(20)

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows
FROM fq_dev_catalog.silver.fact_financial_pnl;

In [0]:
%sql
select * from fq_dev_catalog.silver.fact_financial_pnl limit 1

In [0]:
for query in spark.streams.active:
    query.stop()